# Lekcia 02 - Preskúmanie Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** je jednotný rámec na tvorbu AI agentov. Poskytuje čistú, skladateľnú architektúru so štyrmi základnými stavebnými blokmi:

- **Klient** – pripája sa na koncový bod AI modelu a spravuje komunikáciu
- **Agent** – obalí klienta so súbormi inštrukcií a definíciami nástrojov
- **Nástroje** – rozširujú schopnosti agenta o vlastné funkcie, ktoré môže model volať
- **Sedenie** – udržuje históriu konverzácie pre viacstupňové interakcie

V tejto lekcii vytvoríme **agenta na rezerváciu ciest**, ktorý kontroluje dostupnosť cieľových destinácií pomocou týchto konceptov.


## Nastavenie


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## Pochopenie architektúry Agenta Frameworku

Microsoft Agent Framework nasleduje vrstvenú architektúru:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Klient** – `FoundryChatClient` sa pripája k Azure OpenAI nasadeniu. Zabezpečuje autentifikáciu, formátovanie požiadaviek a parsovanie odpovedí.
2. **Agent** – Vytvorený z klienta cez `provider.create_agent()`, agent kombinuje prístup k modelu s inštrukciami (systémovým promptom) a nástrojmi.
3. **Nástroje** – Python funkcie dekorované `@tool`, ktoré agent môže vyvolať na vykonávanie akcií alebo získavanie dát.
4. **Relácia** – Objekt `AgentSession` (vytvorený cez `agent.create_session()`), ktorý uchováva históriu konverzácie, umožňujúc viackolový dialóg, kde si agent pamätá predchádzajúci kontext.

Poďme zostaviť každú vrstvu krok za krokom.


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Pridanie nástrojov pomocou dekorátora @tool

Nástroje umožňujú agentom vykonávať akcie nad rámec generovania textu. Dekorátor `@tool` prevádza bežnú Python funkciu na niečo, čo môže agent volať.

Kľúčové body:
- Použite `Annotated[type, "description"]`, aby model pochopil každý parameter.
- Docstring sa stáva popisom nástroja, ktorý model vidí.
- `approval_mode="never_require"` znamená, že nástroj sa spustí automaticky bez potvrdenia používateľa.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Vytvorenie agenta s nástrojmi

Teraz skombinujeme klienta, inštrukcie a nástroje do agenta. `instructions` slúžia ako systémový prompt — definujú osobnosť a správanie agenta.


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Viackolové konverzácie so session

`AgentSession` (vytvorená cez `agent.create_session()`) sleduje všetky správy v konverzácii. Pri každom volaní `agent.run()` s rovnakou session má agent prístup k celej histórii konverzácie a môže sa odvolávať na predchádzajúce správy.

Prechádzame `tools=[check_destination_availability]`, aby agent mohol počas každého kola volať náš kontrolór dostupnosti.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Zhrnutie

V tejto lekcii ste preskúmali štyri piliere Microsoft Agent Framework:

| Koncept | Čo ste sa naučili |
|---------|------------------|
| **Klient** | `FoundryChatClient` sa pripája k Azure OpenAI s autentifikáciou na základe poverení |
| **Agent** | `provider.create_agent()` spája pripojenie modelu s inštrukciami a menom |
| **Nástroje** | Dekorátor `@tool` sprístupňuje Python funkcie, ktoré môže agent volať |
| **Relácia** | `agent.create_session()` udržiava históriu konverzácie cez viaceré kolá |

Tieto stavebné bloky sa spájajú, aby vytvorili agentov, ktorí môžu viesť prirodzené rozhovory, volať externé funkcie a udržiavať kontext — základ pre pokročilejšie agentné vzory v nasledujúcich lekciách.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vyhlásenie o zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, vezmite prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho natívnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
